# Week 4: Transfer Learning, BERT (Seminar)

### Using pretrained transformers (for fun, profit and 1 point)

There are many toolkits that let you access pretrained transformer models (like we used pretrained embeddings earlier), but the most powerful and convenient by far is 🤗[`huggingface/transformers`](https://github.com/huggingface/transformers). In this week's practice, you'll learn how to download, apply and modify pretrained transformers for a range of tasks. Buckle up, we're going in!


__Pipelines:__ if all you want is to apply a pretrained model, you can do that in one line of code using pipeline. Huggingface/transformers has a selection of pre-configured pipelines for masked language modelling, sentiment classification, question aswering, etc. ([see full list here](https://huggingface.co/transformers/main_classes/pipelines.html))

A typical pipeline includes:
* pre-processing, e.g. tokenization, subword segmentation
* a backbone model, e.g. bert finetuned for classification
* output post-processing

Let's see it in action:

In [1]:
import os
from IPython.display import clear_output

notebook_dir = "/home/balabaevvl/courses/nlp/nlp_course/week04_transfer"  # notebook's dir
os.chdir(notebook_dir)

print("Current dir:", os.getcwd())


Current dir: /home/balabaevvl/courses/nlp/nlp_course/week04_transfer


In [2]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-fe2d8dfd-06f2-a5c4-a7fd-4a5f23947005"     # 2
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-0c320096-21ee-4060-8731-826ca2febfab"     # 3
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-baef952c-6609-aace-3b78-e4e07788d5de"     # 4
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3979d65b-c238-4e9c-0c1c-1aa3f05c56a1"     # 5

import torch
device = torch.device('cuda:0')

In [3]:
import shutil, os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"    # problems with progress bar

from huggingface_hub.constants import HF_HUB_CACHE
cache = os.path.expanduser(HF_HUB_CACHE)
for d in os.listdir(cache):
    if d.startswith("models--distilbert-base-uncased-finetuned-sst-2-english"):
        shutil.rmtree(os.path.join(cache, d), ignore_errors=True)

# from huggingface_hub import snapshot_download
# local_dir = snapshot_download("distilbert-base-uncased-finetuned-sst-2-english")


In [4]:
import transformers

In [5]:
sentiment_clf = transformers.pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)
# clear_output()

sentiment_clf(["transformers library can be really useful!", "YSDA midterm is soon"])

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9959487915039062},
 {'label': 'NEGATIVE', 'score': 0.986364483833313}]

In [6]:
transformers.pipelines.SUPPORTED_TASKS.keys()

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'summarization', 'translation', 'text2text-generation', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-to-text', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'image-to-image', 'keypoint-matching'])

But how can we find out which model is suitable for chosen task in such a big models space?

Option 1: Using search and filters in [web](https://huggingface.co/models) (user-friendly)

Option 2: Using `huggingface_hub` library to access API from Python (if you want to automate some process)


In [7]:
import huggingface_hub

In [8]:
some_model = next(huggingface_hub.list_models())

some_model

ModelInfo(id='deepseek-ai/DeepSeek-OCR', author=None, sha=None, created_at=datetime.datetime(2025, 10, 17, 6, 22, 5, tzinfo=datetime.timezone.utc), last_modified=None, private=False, disabled=None, downloads=302402, downloads_all_time=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, likes=1530, library_name=None, tags=['safetensors', 'deepseek_vl_v2', 'deepseek', 'vision-language', 'ocr', 'custom_code', 'image-text-to-text', 'multilingual', 'arxiv:2510.18234', 'license:mit', 'region:us'], pipeline_tag='image-text-to-text', mask_token=None, card_data=None, widget_data=None, model_index=None, config=None, transformers_info=None, trending_score=1530, siblings=None, spaces=None, safetensors=None, security_repo_status=None, xet_enabled=None)

In [9]:
filter = (
    "sentiment-analysis",
    "pytorch",
    "ru",
)

filtered_models = huggingface_hub.list_models(
    filter=filter,
    sort="downloads",
    limit=10,
)

print(f"Filtered by {filter}:")
for model in filtered_models:
    print(f"- https://huggingface.co/{model.id} ({model.downloads} downloads, {model.likes} likes)")

Filtered by ('sentiment-analysis', 'pytorch', 'ru'):
- https://huggingface.co/seara/rubert-tiny2-russian-sentiment (146400 downloads, 29 likes)
- https://huggingface.co/yangheng/deberta-v3-base-absa-v1.1 (115448 downloads, 60 likes)
- https://huggingface.co/r1char9/rubert-base-cased-russian-sentiment (6778 downloads, 12 likes)
- https://huggingface.co/yangheng/deberta-v3-large-absa-v1.1 (439 downloads, 20 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-ru-go-emotions (377 downloads, 9 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-cedr (256 downloads, 3 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-ru-go-emotions (197 downloads, 4 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-sentiment (156 downloads, 11 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-cedr (73 downloads, 1 likes)
- https://huggingface.co/oxygeneDev/sentiment-multilingual (61

Imagine the situation when you have a long text to read and a lack of time. Luckily, you've got an option to use one of pipelines! But which one?...

**Task 1 (0.5 points)**
- Find a suitable pipeline and model for text below
- Apply model to long text to get a short one
- Pretty-print the result and give an opinion if short text is good or not



In [10]:
filtered_models = huggingface_hub.list_models(
    filter=("en", "summarization", "pytorch",),
    sort="downloads",
    limit=10,
)

for model in filtered_models:
    print(f"- https://huggingface.co/{model.id} ({model.downloads} downloads, {model.likes} likes)")

- https://huggingface.co/google-t5/t5-small (3715023 downloads, 497 likes)
- https://huggingface.co/facebook/bart-large-cnn (2705450 downloads, 1483 likes)
- https://huggingface.co/google-t5/t5-base (1413120 downloads, 754 likes)
- https://huggingface.co/google-t5/t5-3b (1296568 downloads, 48 likes)
- https://huggingface.co/sshleifer/distilbart-cnn-12-6 (876080 downloads, 297 likes)
- https://huggingface.co/google-t5/t5-large (255046 downloads, 223 likes)
- https://huggingface.co/philschmid/bart-large-cnn-samsum (176623 downloads, 264 likes)
- https://huggingface.co/sshleifer/distilbart-xsum-12-6 (89900 downloads, 7 likes)
- https://huggingface.co/google/pegasus-xsum (89279 downloads, 212 likes)
- https://huggingface.co/Falconsai/text_summarization (33077 downloads, 266 likes)


In [11]:
summarization_clf = transformers.pipeline(
    task="summarization",
    model="google-t5/t5-small",
)


<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
Device set to use cuda:0


In [12]:
long_text = """
The widespread adoption of remote work, accelerated by global events in the early 2020s, has triggered a significant and likely permanent shift in how we think about the workplace. This transition away from the traditional central office is having profound and multifaceted effects on urban economies, reshaping everything from commercial real estate to local small businesses.

One of the most immediate and visible impacts has been on the commercial real estate sector. With companies downsizing their physical footprints or adopting fully remote models, demand for office space has plummeted. This has led to rising vacancy rates, downward pressure on commercial rent prices, and a re-evaluation of the financial viability of large office buildings. City governments, which often rely heavily on property taxes from these high-value commercial properties, are now facing substantial budget shortfalls.

Furthermore, the daily rhythm of city centers has changed dramatically. The decline in the number of commuters has had a ripple effect on local businesses that once thrived on their patronage. Lunchtime cafes, after-work bars, dry cleaners, and public transit systems have all experienced a significant drop in revenue. This "doughnut effect" describes a phenomenon where the economic activity hollows out in the city center and increases in suburban residential areas as people work from home and spend their money locally.

However, it's not all negative. This shift also presents new opportunities. Some urban planners see a chance to repurpose vacant office buildings into much-needed residential housing, which could help address housing shortages and revitalize neighborhoods by creating 24/7 communities. Additionally, the ability to work remotely has spurred a reversal of rural depopulation in some regions, as professionals seek a better quality of life outside of major metropolitan areas, potentially distributing economic growth more evenly.

In conclusion, the remote work revolution is fundamentally restructuring urban economies. While it presents serious challenges to established systems like commercial real estate and downtown commerce, it also opens the door to innovative urban renewal and a more geographically dispersed economic landscape. The long-term effects will depend on how effectively cities and businesses can adapt to this new, more flexible paradigm.
"""

short_text = summarization_clf([long_text])[0]["summary_text"]
print(short_text)

the widespread adoption of remote work has triggered a significant and likely permanent shift in how we think about the workplace . the shift is reshaping everything from commercial real estate to local small businesses . this shift also presents new opportunities for urban planners to repurpose vacant office buildings .


In [13]:
assert len(long_text) / len(short_text) > 5, "Too long, didn't read"

One of possible semi-supervised tasks used while BERT training is Masked Language Modeling. So our model have some text prediction capabilities!



In [14]:
mlm_model = transformers.pipeline(
    task="fill-mask",
    model="bert-base-cased",
)

mlm_model("My name is [MASK] Shady!")

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cuda:0


[{'score': 0.09115735441446304,
  'token': 22191,
  'token_str': 'Slim',
  'sequence': 'My name is Slim Shady!'},
 {'score': 0.02567778155207634,
  'token': 2791,
  'token_str': 'Captain',
  'sequence': 'My name is Captain Shady!'},
 {'score': 0.023223962634801865,
  'token': 3056,
  'token_str': 'Miss',
  'sequence': 'My name is Miss Shady!'},
 {'score': 0.01629173755645752,
  'token': 4479,
  'token_str': 'Jimmy',
  'sequence': 'My name is Jimmy Shady!'},
 {'score': 0.013304653577506542,
  'token': 13960,
  'token_str': 'Mister',
  'sequence': 'My name is Mister Shady!'}]

In order to make result more readable we can just take top-1 result:

In [15]:
mlm_model("My name is [MASK] Shady!")[0]["sequence"]

'My name is Slim Shady!'

**Task 2 (0.5 points)**
- Using BERT's ability to solve MLM task, find out answers on the following questions
- Perform some fact-checking, don't trust LLMs!

**Questions:**
- When YSDA was founded?
- Who invented radio first?
- What is the fifth Fibonacci number?

In [16]:
get_possible_answers = lambda x: [i["token_str"] for i in mlm_model(x)]

# '2007'
print(get_possible_answers("YSDA (Yandex School of Data Analysis) in Russia was founded in [MASK] year."))
print(get_possible_answers("The two-year Yandex program was created in [MASK] and has become Russia’s leading data analysis program. Courses from the Yandex School of Data Analysis serve as the foundation for Master’s programs at major universities, such as the Higher School of Economics and the Moscow Institute of Physics and Technology."))

# '3'
print('----\n')
print(get_possible_answers("Fifth Fibonacci number is [MASK]"))
print(get_possible_answers("The Fibonacci numbers 0 1 1 2 and the next term is 1+2=3 so we now have 0 1 1 2 3 and it continues as follows ... 0, 1, 1, 2, [MASK], 5, 8, 13, 21, 34, 55, 89, 144, ..."))

# 'Edison'
print('----\n')
print(get_possible_answers("The radio was inverted by [MASK]"))
print(get_possible_answers("In the mid-1890s, building on techniques physicists were using to study electromagnetic waves, [MASK] developed the first apparatus for long-distance radio communication.[1] On 23 December 1900, the Canadian-born American inventor Reginald A. Fessenden became the first person to send audio (wireless telephony) by means of electromagnetic waves, successfully transmitting over a distance of about a mile (1.6 kilometers,) and six years later on Christmas Eve 1906 he became the first person to make a public wireless broadcast."))


['that', 'this', 'same', 'last', 'the']
['2005', '1999', '1995', '2002', '2003']
----

['.', ';', '?', ':', '!']
['3', '4', '2', '1', '5']
----

['.', ';', '!', '?', ':']
['he', 'they', 'scientists', 'researchers', 'Edison']


---

### The building blocks of a pipeline

Huggingface also allows you to access its pipelines on a lower level. There are two main abstractions for you:
* `Tokenizer` - converts from strings to token ids and back
* `Model` - a PyTorch `nn.Module` with pretrained weights

You can use such models as part of your regular PyTorch code: insert it as a layer in your model, apply to a batch of data, backpropagate, optimize, etc.

In [17]:
tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
model = transformers.AutoModel.from_pretrained("bert-base-uncased")

In [18]:
lines = [
    "Luke, I am your father.",
    "Life is what happens when you're busy making other plans.",
    "I have no idea what pneumonoultramicroscopicsilicovolcanoconiosis is."
]

tokens_info = tokenizer(lines, padding=True, truncation=True, return_tensors="pt")
print("Tokenized:")
print(tokens_info)

print("\nDetokenized:")
for i in range(3):
    print(tokenizer.decode(tokens_info['input_ids'][i]))

Tokenized:
{'input_ids': tensor([[  101,  5355,  1010,  1045,  2572,  2115,  2269,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2166,  2003,  2054,  6433,  2043,  2017,  1005,  2128,  5697,
          2437,  2060,  3488,  1012,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1045,  2031,  2053,  2801,  2054,  1052,  2638,  2819, 17175,
         11314,  6444,  2594,  7352, 26461, 27572, 11261,  6767, 15472,  6761,
          8663, 10735,  2483,  2003,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1,

You can see some special tokens appeared besides our original text. They are usually used to give model some additional information, so model treats them in individual way.

You can list all special tokens used by tokenizer (moreover, you can add your own special tokens, but make sure you will show them to your model while training):

In [19]:
tokenizer._special_tokens_map

{'bos_token': None,
 'eos_token': None,
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]',
 'additional_special_tokens': []}

In [20]:
tokenizer("First sentence", "Second sentence", return_token_type_ids=True)

{'input_ids': [101, 2034, 6251, 102, 2117, 6251, 102], 'token_type_ids': [0, 0, 0, 0, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

It's ineffective to put all possible tokens in vocabulary, but one also want to handle all possible text sequences instead of putting UNK everywhere.

WordPiece tokenization is here to help!

In [21]:
reversed_vocab = {token_id: token for token, token_id in tokenizer.vocab.items()}

In [22]:
for token_id in tokens_info["input_ids"][2]:
    print(reversed_vocab[token_id.item()], end=' ')

[CLS] i have no idea what p ##ne ##um ##ono ##ult ##ram ##ic ##ros ##copic ##sil ##ico ##vo ##lc ##ano ##con ##ios ##is is . [SEP] 

Now you can apply tokenized data with model.

Depending on your task, you can use different part of output. For example, `[CLS]`-token output can be obtained by `pooler_output` key in model output.

In [23]:
import torch

In [24]:
with torch.no_grad():
    out = model(**tokens_info)

print(out['pooler_output'])

tensor([[-0.8854, -0.4722, -0.9392,  ..., -0.8081, -0.6955,  0.8748],
        [-0.9297, -0.5161, -0.9334,  ..., -0.9017, -0.7492,  0.9201],
        [-0.6808, -0.1979, -0.7096,  ..., -0.6691, -0.4557,  0.7595]])


Transformers knowledge hub: https://huggingface.co/transformers/



---



### Visualizing BERT

Interpretability of models is one of key factors of understanding their behaviour.

Neural Networks are harder to interpret than Classic ML models, but still it's not impossible!

Remember Attention mechanism? It's human-understandable concept: look closely to tokens which are more valuable for context of the current one.

In [25]:
from transformers import AutoTokenizer, AutoModel, utils
from bertviz import model_view, head_view

input_text = "Every time I try to interpret BERT model behaviour, I find new interesting patterns"
model = AutoModel.from_pretrained("bert-base-cased", output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

inputs = tokenizer.encode(input_text, return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/bertviz/model_view.py:232: ResourceWarning: unclosed file <_io.TextIOWrapper name='/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/bertviz/model_view.js' mode='r' encoding='UTF-8'>
  vis_js = open(os.path.join(__location__, 'model_view.js')).read().replace("PYTHON_PARAMS", json.dumps(params))


<IPython.core.display.Javascript object>

In [26]:
head_view(attention, tokens)

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/bertviz/head_view.py:220: ResourceWarning: unclosed file <_io.TextIOWrapper name='/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/bertviz/head_view.js' mode='r' encoding='UTF-8'>
  vis_js = open(os.path.join(__location__, 'head_view.js')).read().replace("PYTHON_PARAMS", json.dumps(params))


<IPython.core.display.Javascript object>

Another possible task for BERT training is Next Sentence Prediction.

How BERT's heads looks at tokens in that case?

In [27]:
inputs = tokenizer.encode("I'm waiting for important call", "I can't go out right now", return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

<IPython.core.display.Javascript object>

In [28]:
head_view(attention, tokens)

<IPython.core.display.Javascript object>

It looks interesting, doesn't it?

If you want to find out more about attention patterns, you can refer to special "field" of science - [BERTology](https://huggingface.co/docs/transformers/main/en/bertology).



---



### Tuning pretrained transfomers (for your own task and 2 points)

Important benefit of using big models is their ability to adapt to various tasks without spending a lot of time and resources for full training.

You could've heard about backbone models in another ML tasks, when they're tuned using specific data.

It's possible to tune model's weights directly, but you also can freeze model, use its outputs as knowledge and then extract neccessary information using much smaller neural networks.

#### Introduction

Here's an example of tuned BERT base model for Named Entity Recognition (NER) task:

In [29]:
tokenizer = transformers.AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = transformers.AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [30]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

As you can see, there's an additional classifier besides original BERT content. That layer is used to predict NER-classes for each BERT's token output.

BERT is suitable for tuning for different tasks since it outputs token embeddings and the whole data embedding in `[CLS]`-token as well.

#### Data preparation

In [31]:
import datasets

In [32]:
dataset = datasets.load_dataset("lhoestq/conll2003")

In [33]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [34]:
dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

Since BERT tokenization is different from the dataset's one, we need to fix that divergence.

**Task 3 (0.5 points)**
- Align dataset token labels to WordPiece tokens
- Handle special tokens as well

In [35]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification


tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(samples, pad_label: int = 0):
    tokenized_inputs = tokenizer(samples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, original_labels in enumerate(samples["ner_tags"]):
        step_labels = [pad_label]
        
        for j, token in enumerate(samples["tokens"][i]):
            step_labels += [original_labels[j]] * len(tokenizer.tokenize(token))
        
        step_labels.append(pad_label)
        labels.append(step_labels)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [36]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [37]:
tokenized_dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [101,
  7270,
  22961,
  1528,
  1840,
  1106,
  21423,
  1418,
  2495,
  12913,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [0, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, 0]}

In [38]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    return_tensors="pt"  # PyTorch tensors
)


Now dataset is ready to be used by BERT.

#### Model preparation

For our task we can use `AutoModelForTokenClassification`, which already provides required architecture with token classifier (e.g. classifier itself, class outputs).

You can handle these things by yourself: create PyTorch model class, init BERT model and Linear layer for classification, then override forward method and so on...

`AutoModelForTokenClassification` is chosen for the sake of simplicity, but it's still required for MLE to be capable of doing it with bare hands.

In [39]:
from transformers import AutoModelForTokenClassification

id2label = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG", 5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}
label2id = {label: id for id, label in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=9,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Evaluation

Evaluation is crucial while writing papers or reporting your work results. Sometimes it can be tricky and own implementation can be buggy, so it usually preferred to calculate metrics using frameworks.

Let's prepare `compute_metrics` function for the following training loop:

In [40]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    y_true = []
    y_pred = []
    for i in range(len(predictions)):
        y_true_sample = []
        y_pred_sample = []
        for j in range(len(predictions[i])):
            if labels[i][j] == -100:
                continue

            y_true_sample.append(id2label[int(labels[i][j])])
            y_pred_sample.append(id2label[int(predictions[i][j])])

        y_true.append(y_true_sample)
        y_pred.append(y_pred_sample)

    return {
        "precision": precision_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "recall": recall_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "f1": f1_score(y_true, y_pred, mode="strict", scheme=IOB2),
    }

#### Training

**Task 4 (0.5 points)**
- Choose proper hyperparameters for tuning the model
- Setup HF Trainer
- Check correctness using training results


In [41]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="torch.utils.data")


In [42]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-ner",
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    logging_dir="./logs",
    report_to="none",
    learning_rate=1e-4,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Let's check metrics before training:

In [44]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

{'eval_loss': 2.2893028259277344, 'eval_model_preparation_time': 0.0265, 'eval_precision': 0.014897663466763267, 'eval_recall': 0.03052231190277391, 'eval_f1': 0.020022517725101177, 'eval_runtime': 20.3952, 'eval_samples_per_second': 169.305, 'eval_steps_per_second': 10.591}


In [45]:
trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
100,0.326600,0.158255,0.026500,0.817440,0.799839,0.808543
200,0.141200,0.100891,0.026500,0.894583,0.877086,0.885748
300,0.133400,0.104002,0.026500,0.901842,0.882828,0.892234
400,0.092200,0.079154,0.026500,0.904399,0.905616,0.905007
500,0.089100,0.076158,0.026500,0.927661,0.907770,0.917608
600,0.071600,0.063743,0.026500,0.920079,0.921317,0.920698
700,0.063800,0.058991,0.026500,0.931607,0.926341,0.928967
800,0.052400,0.054163,0.026500,0.936193,0.931994,0.934089


TrainOutput(global_step=878, training_loss=0.1158752848727285, metrics={'train_runtime': 265.5338, 'train_samples_per_second': 52.878, 'train_steps_per_second': 3.307, 'total_flos': 351240792638148.0, 'train_loss': 0.1158752848727285, 'epoch': 1.0})

In [46]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)
# 'eval_f1': 0.892

{'eval_loss': 0.12765589356422424, 'eval_model_preparation_time': 0.0265, 'eval_precision': 0.8938666174025495, 'eval_recall': 0.8977641710733835, 'eval_f1': 0.895811154825272, 'eval_runtime': 16.3684, 'eval_samples_per_second': 210.955, 'eval_steps_per_second': 13.196, 'epoch': 1.0}


Compare test metrics before and after training. Did we succeed?

**Task 5 (1 point)**
- Compare our model's result with `dslim/bert-base-NER`
- Try to improve our model's quality. Choose any option:
  - Play with training hyperparameters (batch_size, lr, epochs, etc.)
  - Apply some training techniques (warm-up, lr-scheduling, etc.)
  - Perform error analysis and find model's weak spots (this option doesn't require fixing them)
  - Your very own idea
- Write a small report (up to 5 steps, results and conclusions) on the work done in "Tuning pretrained transformers" part

In [47]:
upd_model = transformers.AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [48]:
upd_training_args = TrainingArguments(
    output_dir="./bert-ner",
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    logging_dir="./logs",
    report_to="none",
    learning_rate=1e-4,
    num_train_epochs=4,             # <- helps
    per_device_train_batch_size=24, # <- tried 32, CUDA was out of memory (even with (b)fp16)
    per_device_eval_batch_size=24,  # <- tried 32, CUDA was out of memory (even with (b)fp16)
    # fp16=True,                      # <- [if CUDA] tried, was worse in test
    bf16=True,                      # [if GPU supports BF16 (A100/H100/RTX 3xxx+)] tried, was worse in test
    warmup_ratio=0.1,               # <- helps
)

# tried different combos

In [49]:
upd_trainer = Trainer(
    model=upd_model,
    args=upd_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [50]:
results = upd_trainer.evaluate(tokenized_dataset["test"])
print(results)

{'eval_loss': 0.2028985470533371, 'eval_model_preparation_time': 0.0478, 'eval_precision': 0.8575421512938418, 'eval_recall': 0.8823638556452361, 'eval_f1': 0.8697759487882945, 'eval_runtime': 20.6059, 'eval_samples_per_second': 167.573, 'eval_steps_per_second': 6.988}


In [ ]:
upd_trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
100,0.028200,0.076947,0.047800,0.920964,0.929392,0.925159
200,0.029800,0.091832,0.047800,0.886831,0.897811,0.892287
300,0.037400,0.073125,0.047800,0.927123,0.910820,0.918899
400,0.043900,0.095339,0.047800,0.910623,0.915934,0.913271
500,0.043700,0.086893,0.047800,0.916477,0.901758,0.909058
600,0.058000,0.082947,0.047800,0.922959,0.921138,0.922048
700,0.031800,0.080271,0.047800,0.928565,0.911986,0.920201
800,0.030500,0.078641,0.047800,0.924917,0.922842,0.923878
900,0.027100,0.074155,0.047800,0.934507,0.924278,0.929364
1000,0.029600,0.069494,0.047800,0.919943,0.929930,0.924910


TrainOutput(global_step=2344, training_loss=0.02136639515160497, metrics={'train_runtime': 1069.132, 'train_samples_per_second': 52.532, 'train_steps_per_second': 2.192, 'total_flos': 1492793528624406.0, 'train_loss': 0.02136639515160497, 'epoch': 4.0})

In [ ]:
results = upd_trainer.evaluate(tokenized_dataset["test"])
print(results)

{'eval_loss': 0.18861420452594757, 'eval_model_preparation_time': 0.0478, 'eval_precision': 0.8943830570902395, 'eval_recall': 0.9011039985156323, 'eval_f1': 0.8977309487499423, 'eval_runtime': 19.4242, 'eval_samples_per_second': 177.768, 'eval_steps_per_second': 7.413, 'epoch': 4.0}


In [ ]:
# check out TrainingArguments comments

NameError: name 'Report' is not defined